In [1]:
# Cell 1 - Verificar acceso a los archivos en Bronze
import os

# Listar archivos disponibles en el Lakehouse
for year in ['AW2019', 'AW2022', 'AW2025']:
    for schema in ['Sales', 'Production', 'Purchasing']:
        path = f"Files/{year}/AW_{schema}_{year[2:]}/{schema}"
        try:
            files = mssparkutils.fs.ls(f"Files/{year}/AW_{schema}_{year[2:]}/{schema}")
            print(f"{year}/{schema}: {len(files)} archivos")
        except Exception as e:
            print(f"ERROR {year}/{schema}: {e}")

StatementMeta(, 361b8e56-5f82-474b-be3c-23875eebf3c9, 3, Finished, Available, Finished, False)

AW2019/Sales: 5 archivos
AW2019/Production: 5 archivos
AW2019/Purchasing: 2 archivos
AW2022/Sales: 5 archivos
AW2022/Production: 5 archivos
AW2022/Purchasing: 2 archivos
AW2025/Sales: 5 archivos
AW2025/Production: 5 archivos
AW2025/Purchasing: 2 archivos


In [2]:
# Cell 2 - Leer todos los parquet y registrarlos como tablas Delta en Bronze
from pyspark.sql.functions import lit

# Configuración de rutas
TABLES = {
    "Sales": [
        "SalesOrderHeader",
        "SalesOrderDetail",
        "Customer",
        "SalesTerritory",
        "SalesPerson"
    ],
    "Production": [
        "Product",
        "ProductCategory",
        "ProductSubcategory",
        "ProductInventory",
        "WorkOrder"
    ],
    "Purchasing": [
        "PurchaseOrderHeader",
        "Vendor"
    ]
}

YEARS = {
    "AW2019": "2019",
    "AW2022": "2022",
    "AW2025": "2025"
}

# Leer y guardar cada tabla como Delta
for folder, year in YEARS.items():
    for schema, tables in TABLES.items():
        for table in tables:
            path = f"Files/{folder}/AW_{schema}_{year}/{schema}/{table}.parquet"
            
            try:
                df = spark.read.parquet(path)
                
                # Agregar columna de año para identificar la fuente
                df = df.withColumn("source_year", lit(year))
                
                # Nombre de tabla en el Lakehouse: bronze_Sales_SalesOrderHeader_2019
                table_name = f"bronze_{schema}_{table}_{year}"
                
                df.write.format("delta").mode("overwrite").saveAsTable(table_name)
                
                print(f"OK: {table_name} ({df.count():,} filas)")
                
            except Exception as e:
                print(f"ERROR: {schema}.{table} {year} -> {e}")

StatementMeta(, 361b8e56-5f82-474b-be3c-23875eebf3c9, 4, Finished, Available, Finished, False)

OK: bronze_Sales_SalesOrderHeader_2019 (31,465 filas)
OK: bronze_Sales_SalesOrderDetail_2019 (121,317 filas)
OK: bronze_Sales_Customer_2019 (19,820 filas)
OK: bronze_Sales_SalesTerritory_2019 (10 filas)
OK: bronze_Sales_SalesPerson_2019 (17 filas)
OK: bronze_Production_Product_2019 (504 filas)
OK: bronze_Production_ProductCategory_2019 (4 filas)
OK: bronze_Production_ProductSubcategory_2019 (37 filas)
OK: bronze_Production_ProductInventory_2019 (1,069 filas)
OK: bronze_Production_WorkOrder_2019 (72,591 filas)
OK: bronze_Purchasing_PurchaseOrderHeader_2019 (4,012 filas)
OK: bronze_Purchasing_Vendor_2019 (104 filas)
OK: bronze_Sales_SalesOrderHeader_2022 (31,465 filas)
OK: bronze_Sales_SalesOrderDetail_2022 (121,317 filas)
OK: bronze_Sales_Customer_2022 (19,820 filas)
OK: bronze_Sales_SalesTerritory_2022 (10 filas)
OK: bronze_Sales_SalesPerson_2022 (17 filas)
OK: bronze_Production_Product_2022 (504 filas)
OK: bronze_Production_ProductCategory_2022 (4 filas)
OK: bronze_Production_ProductS

In [1]:
# Cell 3 - Verificación final
tables = spark.catalog.listTables()
print(f"Total tablas en LH_Bronze: {len(tables)}")
print("\nEjemplo - SalesOrderHeader 2019 vs 2025:")
df_2019 = spark.table("bronze_Sales_SalesOrderHeader_2019")
df_2025 = spark.table("bronze_Sales_SalesOrderHeader_2025")
print(f"2019: {df_2019.count():,} filas | columnas: {len(df_2019.columns)}")
print(f"2025: {df_2025.count():,} filas | columnas: {len(df_2025.columns)}")

StatementMeta(, 15fa6ed3-759e-4075-b2bf-b16783e2eae4, 3, Finished, Available, Finished, False)

Total tablas en LH_Bronze: 36

Ejemplo - SalesOrderHeader 2019 vs 2025:
2019: 31,465 filas | columnas: 27
2025: 31,465 filas | columnas: 27
